# 🧠 MemoRIA — Memoria Personal Semántica

> **Proyecto:** Engineering Project — Curso NLP, MIA, UdeSA 2026
> **Equipo:** Nico Karagozian, Clara Kearney, Valen Pivotto
> **Profesor:** Luciano Del Corro
> **Modelo base:** `Qwen/Qwen3-4B-Instruct-2507` (~4B params, context 32K, soporte nativo de role `system`)

---

## Pregunta de investigación

> ¿Puede un modelo de 4B parámetros aprender múltiples registros lingüísticos de un individuo usando sus propios datos reales y hardware de consumo?

**MemoRIA** es un LLM ajustado fino sobre textos personales en tres registros:

| Tag | Registro | Fuente |
|-----|----------|---------|
| `[CASUAL]` | Conversacional / WhatsApp | Exports `.txt` |
| `[EMAIL-PROF]` | Profesional / email | `.mbox` de Google Takeout |
| `[ACADÉMICO]` | Académico / ensayos | PDFs y DOCX personales |

**Lo que NO es:** No es prompting a un LLM grande. Es fine-tuning real con **LoRA**.

---

## Índice
1. [Setup e instalación](#setup)
2. [Datos y preprocesamiento](#datos)
3. [Fine-tuning](#finetuning)
4. [Inferencia local](#inferencia)
5. [Exportar a Ollama](#ollama)
6. [Evaluación](#evaluacion)
7. [Backend FastAPI](#backend)

---
## 1. Setup e Instalación <a id='setup'></a>

Path único: **MLX-LM con cuantización 4-bit** sobre Mac Apple Silicon. Requiere `mlx-lm>=0.22`.

> ⚠️ Cerrar browsers y apps pesadas antes de entrenar para liberar RAM unificada (el modelo cuantizado pesa ~2.5 GB y el training necesita espacio para gradientes y activaciones).

In [ ]:
# Instalar dependencias (correr una sola vez)
!pip install -q \
    mlx-lm>=0.22.0 \
    transformers>=4.50.0 \
    peft>=0.14.0 \
    trl>=0.12.0 \
    accelerate>=0.27.0 \
    datasets>=2.20.0 \
    pdfplumber>=0.10.0 \
    python-docx>=1.1.0 \
    html2text>=2024.2.26 \
    nltk>=3.8.0 \
    scikit-learn>=1.4.0 \
    scipy>=1.12.0 \
    tqdm>=4.66.0 \
    httpx>=0.27.0

In [ ]:
import os
import sys
import re
import json
import math
import random
import mailbox
from pathlib import Path
from collections import Counter
from email.header import decode_header

# MPS fallback: operaciones sin kernel MPS nativo caen a CPU en vez de crashear
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import torch
import numpy as np

print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
print(f"MPS disponible:  {torch.backends.mps.is_available()}")

if torch.cuda.is_available():
    print(f"GPU CUDA: {torch.cuda.get_device_name(0)} — {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif torch.backends.mps.is_available():
    print("GPU Metal (Apple Silicon) disponible")

# Agregar el root del proyecto al path para importar scripts/
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
# ─── Configuración global ────────────────────────────────────────────────────

# Cambiar estos valores antes de correr
AUTHOR_NAME  = "Nicolas Karagozian"        # Nombre exacto en WhatsApp
AUTHOR_EMAIL = "nkaragozian@tnplatex.com"  # Email del remitente en Gmail

# Modelo base — Qwen 3 4B Instruct
MODEL_ID       = "Qwen/Qwen3-4B-Instruct-2507"
MLX_MODEL_PATH = Path("./models/qwen3-4b-4bit")  # Modelo cuantizado local

DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

# Rutas
DATA_DIR      = Path("data")
RAW_DIR       = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
DATASET_DIR   = DATA_DIR / "dataset"
OUTPUT_DIR    = Path("memoria-lora")

for d in [RAW_DIR / "whatsapp", RAW_DIR / "gmail", RAW_DIR / "academic",
          PROCESSED_DIR, DATASET_DIR, OUTPUT_DIR, MLX_MODEL_PATH.parent]:
    d.mkdir(parents=True, exist_ok=True)

print("Configuración lista")
print(f"  Modelo: {MODEL_ID}")
print(f"  Device: {DEVICE}")
print(f"  data/raw/whatsapp/ ← copiá tus .txt acá")
print(f"  data/raw/gmail/    ← copiá tu .mbox acá")
print(f"  data/raw/academic/ ← copiá tus PDFs/DOCX acá")

---
## 2. Datos y Pipeline de Preprocesamiento <a id='datos'></a>

| Fuente | Registro | Volumen esperado |
|--------|----------|-----------------|
| WhatsApp (.txt) | Casual | 500–2000 |
| Gmail (.mbox) | Email prof. | 200–800 |
| PDFs/DOCX | Académico | 300–1000 chunks |
| **Total** | | **≥ 1500 ejemplos** |

### Cómo obtener los datos
- **WhatsApp:** Chat → ⋮ → *Exportar chat* → *Sin archivos* → `.txt`
- **Gmail:** myaccount.google.com → *Descargar datos* → Solo Gmail → `.mbox`
- **Académico:** TPs, papers, informes en PDF o DOCX

> ℹ️ **Privacidad:** el modelo se entrena y ejecuta 100% local (Mac + Ollama), nunca se sube a servidores externos. Los textos personales se usan tal cual — sin anonimización de PII — porque nunca salen de la máquina del autor. **No publicar el modelo entrenado.**

In [ ]:
# ─── 2.1 Parser de WhatsApp ───────────────────────────────────────────────────
from scripts.parse_whatsapp import parse_whatsapp

casual_examples = []
for txt_file in (RAW_DIR / "whatsapp").glob("*.txt"):
    print(f"Procesando: {txt_file.name}")
    parsed = parse_whatsapp(str(txt_file), author_name=AUTHOR_NAME)
    casual_examples.extend(parsed)
    print(f"  → {len(parsed)} mensajes")

print(f"\nTotal casual: {len(casual_examples)} ejemplos")

casual_out = PROCESSED_DIR / "casual.jsonl"
with open(casual_out, "w", encoding="utf-8") as f:
    for ex in casual_examples:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")
print(f"Guardado en {casual_out}")

In [ ]:
# ─── 2.2 Parser de Gmail (.mbox) ─────────────────────────────────────────────
from scripts.parse_gmail import parse_mbox

email_examples = []
for mbox_file in (RAW_DIR / "gmail").glob("*.mbox"):
    print(f"Procesando: {mbox_file.name} (puede tardar varios minutos)")
    parsed = parse_mbox(str(mbox_file), sender_email=AUTHOR_EMAIL)
    email_examples.extend(parsed)
    print(f"  → {len(parsed)} emails")

print(f"\nTotal email_prof: {len(email_examples)} ejemplos")

email_out = PROCESSED_DIR / "email_prof.jsonl"
with open(email_out, "w", encoding="utf-8") as f:
    for ex in email_examples:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")
print(f"Guardado en {email_out}")

In [ ]:
# ─── 2.3 Parser académico (PDF / DOCX) ───────────────────────────────────────
from scripts.parse_academic import parse_academic_folder

academic_examples = parse_academic_folder(str(RAW_DIR / "academic"))
print(f"\nTotal academic: {len(academic_examples)} chunks")

academic_out = PROCESSED_DIR / "academic.jsonl"
with open(academic_out, "w", encoding="utf-8") as f:
    for ex in academic_examples:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")
print(f"Guardado en {academic_out}")

In [ ]:
# ─── 2.4 Construcción del dataset final (train / val / test 80/10/10) ─────────
from scripts.build_dataset import build_dataset

train_set, val_set, test_set = build_dataset(
    casual_file=str(PROCESSED_DIR / "casual.jsonl"),
    email_file=str(PROCESSED_DIR / "email_prof.jsonl"),
    academic_file=str(PROCESSED_DIR / "academic.jsonl"),
)

In [ ]:
# Inspección — verificar que el dataset está en formato chat de MLX-LM
print("=" * 70)
print("MUESTRA DEL DATASET (formato messages — MLX-LM aplica chat template)")
print("=" * 70)
for ex in random.sample(train_set, min(3, len(train_set))):
    print(f"\n[{ex['register'].upper()}]")
    for msg in ex["messages"]:
        role = msg["role"].upper()
        content = msg["content"][:200]
        print(f"  {role}: {content}")
    print("─" * 50)
print("\nFormato correcto si cada ejemplo tiene campos: system + user + assistant")


---
## 3. Fine-tuning <a id='finetuning'></a>

**Qwen 3 4B Instruct** tiene ~4B params. En Mac M5 16 GB se **requiere** cuantización 4-bit antes de entrenar — sin cuantización, el modelo no cabe en 16 GB.

| Path | Hardware | Tiempo estimado |
|------|----------|----------------|
| **MLX-LM 4-bit + LoRA** | Mac Apple Silicon 16 GB | ~40–60 min |

> ⚠️ **Importante:** La celda siguiente cuantiza el modelo a 4-bit (paso único, queda en `./models/qwen3-4b-4bit/`). Cerrar browsers y apps pesadas antes de entrenar para liberar RAM unificada.

In [ ]:
# ─── 3.1 Paso 0: cuantizar modelo base a 4-bit ───────────────────────────────
# Correr solo una vez — deja el modelo cuantizado en ./models/qwen3-4b-4bit

if not MLX_MODEL_PATH.exists() or not any(MLX_MODEL_PATH.iterdir()):
    print(f"Descargando y cuantizando {MODEL_ID} a 4-bit...")
    print("(~10-15 min dependiendo del ancho de banda, queda en disco)")
    !{sys.executable} -m mlx_lm convert \
        --hf-path {MODEL_ID} \
        --mlx-path {MLX_MODEL_PATH} \
        --quantize --q-bits 4 --q-group-size 64
    print(f"Modelo cuantizado en {MLX_MODEL_PATH}")
else:
    print(f"Modelo ya existe en {MLX_MODEL_PATH} — saltando conversión")

In [ ]:
# ─── 3.2 Fine-tuning con MLX-LM ──────────────────────────────────────────────
# Tarda ~40-60 min en M5. Monitorear RAM en Activity Monitor.
# Usar --iters 50 para dry-run (verificar que el pipeline compila).

!{sys.executable} -m mlx_lm lora \
    --model {MLX_MODEL_PATH} \
    --train \
    --data data/dataset \
    --fine-tune-type lora \
    --mask-prompt \
    --num-layers -1 \
    --batch-size 1 \
    --learning-rate 5e-5 \
    --iters 900 \
    --val-batches 25 \
    --steps-per-eval 75 \
    --save-every 150 \
    --grad-checkpoint \
    --adapter-path {OUTPUT_DIR} \
    --lora-parameters '{"rank": 16, "alpha": 32, "dropout": 0.05, "scale": 10.0}'
print(f"\nAdaptador LoRA guardado en {OUTPUT_DIR}")

---
## 4. Inferencia local <a id='inferencia'></a>

In [ ]:
# ─── 4. Inferencia con el adaptador entrenado ─────────────────────────────────
# Usa el mismo formato que durante el training: apply_chat_template
# con system prompt + user turn con tag de registro.

from mlx_lm import load, generate as mlx_generate
from pathlib import Path as _P

test_cases = [
    ("casual",     "Contame cómo estuvo el finde"),
    ("email_prof", "Escribí un email pidiendo una reunión para el martes"),
    ("academic",   "Explicá el concepto de fine-tuning en NLP"),
]

print("Cargando modelo + adaptador...")
_model, _tokenizer = load(str(MLX_MODEL_PATH), adapter_path=str(OUTPUT_DIR))

for register, prompt in test_cases:
    system_prompt_path = _P(f"data/system_prompts/{register}.txt")
    system_prompt = system_prompt_path.read_text(encoding="utf-8").strip() if system_prompt_path.exists() else ""
    tag = register.replace("_", "-").upper()
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": f"[{tag}] {prompt}"})
    rendered = _tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    print(f"\n{'='*65}")
    print(f"Registro: {register.upper()} | Prompt: {prompt}")
    print("─"*65)
    output = mlx_generate(
        _model, _tokenizer,
        prompt=rendered,
        max_tokens=300, temp=0.8, top_p=0.9,
        verbose=True,
    )

---
## 5. Exportar a Ollama <a id='ollama'></a>

In [ ]:
MERGED_DIR = Path("memoria-merged")

print("Mergeando con mlx_lm.fuse (dequantize a bf16)...")
!{sys.executable} -m mlx_lm fuse \
    --model {MLX_MODEL_PATH} \
    --adapter-path {OUTPUT_DIR} \
    --save-path {MERGED_DIR} \
    --de-quantize
print(f"Modelo mergeado en {MERGED_DIR} (bf16)")

In [ ]:
import os

merged_abs = os.path.abspath(str(MERGED_DIR))
modelfile_content = f"""FROM {merged_abs}

PARAMETER temperature 0.8
PARAMETER top_p 0.9
PARAMETER repeat_penalty 1.1
PARAMETER num_ctx 4096
"""

with open("Modelfile.local", "w") as f:
    f.write(modelfile_content)

print(f"Modelfile apunta a: {merged_abs}")
!ollama create memoria -f Modelfile.local
!ollama list

---
## 6. Evaluación <a id='evaluacion'></a>

| Nivel | Qué mide | Criterio de éxito |
|-------|----------|-------------------|
| E1 Perplexidad | Probabilidad asignada al texto real | Mejora ≥ 20% vs. base |
| E2 Estilo | TTR, argentinismos, longitud de oraciones | Diff < 20% |
| E3 Clasificador BETO | ¿Se puede distinguir real de generado? | Accuracy < 75% |
| E4 Test ciego humano | Jueces conocidos vs. texto real | % correctas < 65% |

> **Nota metodológica:** E3 y E4 usan prompts del catálogo `data/prompts/` — **no** el inicio del texto real — para medir estilo y no continuación.

In [ ]:
# ─── E1: Perplexidad sobre el TEST SET ───────────────────────────────────────
# Se usa test.jsonl (no val) — separación limpia de evaluación

from eval.perplexity import eval_perplexity

results = eval_perplexity(
    test_file=str(DATASET_DIR / "test.jsonl"),
    model_id=MODEL_ID,
    adapter_path=str(OUTPUT_DIR),
)

In [ ]:
# ─── E2: Métricas de estilo ────────────────────────────────────────────────────
import subprocess, re as _re
from eval.style_metrics import compare_styles

# Cargar textos del test set agrupados por registro
test_by_register = {"casual": [], "email_prof": [], "academic": []}
with open(DATASET_DIR / "test.jsonl") as f:
    for line in f:
        item = json.loads(line)
        if item["register"] in test_by_register:
            test_by_register[item["register"]].append(item["messages"][-1]["content"])

print("Textos en test set por registro:")
for reg, texts in test_by_register.items():
    print(f"  {reg}: {len(texts)}")

# Generar con prompts del catálogo independiente
prompts_casual = [l.strip() for l in open("data/prompts/casual.txt") if l.strip()][:10]

# generate_fn_mlx: usa el modelo mergeado vía subprocess (disponible tras merge-lora)
def generate_fn_mlx(register: str, prompt: str, max_new_tokens: int = 200) -> str:
    _tag = {"casual": "[CASUAL]", "email_prof": "[EMAIL-PROF]", "academic": "[ACADÉMICO]"}.get(register, "[CASUAL]")
    _full = f"{_tag} {prompt}"
    r = subprocess.run(
        [sys.executable, "-m", "mlx_lm", "generate",
         "--model", str(MERGED_DIR),
         "--prompt", _full,
         "--max-tokens", str(max_new_tokens),
         "--temp", "0.8", "--top-p", "0.9"],
        capture_output=True, text=True,
    )
    out = r.stdout.strip()
    if _full in out:
        out = out.split(_full, 1)[-1].strip()
    return _re.sub(r'<[^>]+>', '', out).strip()

generated_casual = [generate_fn_mlx("casual", p) for p in prompts_casual]
real_casual = test_by_register["casual"][:len(generated_casual)]
if real_casual and generated_casual:
    compare_styles(real_casual, generated_casual, register="casual")

In [ ]:
# ─── E3: Clasificador de autoría (BETO) ───────────────────────────────────────
# El clasificador debe FALLAR (accuracy < 75%) → el modelo es indistinguible
# Los textos generados usan prompts del catálogo, NO el inicio del texto real

from eval.train_classifier import train_authorship_classifier

test_texts_all = []
with open(DATASET_DIR / "test.jsonl") as f:
    for line in f:
        test_texts_all.append(json.loads(line)["messages"][-1]["content"])

# generate_fn_mlx definida en la celda eval-style; correr esa celda primero
acc_result = train_authorship_classifier(
    real_texts=test_texts_all,
    generate_fn=generate_fn_mlx,
    n_samples=150,
)

In [ ]:
# ─── E4: Generar pares para test ciego ───────────────────────────────────────
from eval.generate_blind_pairs import generate_blind_test_pairs

pairs = generate_blind_test_pairs(
    test_file=str(DATASET_DIR / "test.jsonl"),
    generate_fn=generate_fn_mlx,
    n_per_register=10,
    output_file="eval/blind_test_pairs.json",
)
print(f"\nArchivo para jueces: eval/blind_test_pairs.json")
print("Ground truth (NO compartir): eval/blind_test_key.json")

In [ ]:
# ─── Resumen de criterios de éxito ────────────────────────────────────────────
import os
from datetime import datetime

print("""
╔══════════════════════════════════════════════════════════╗
║           RESUMEN CRITERIOS DE ÉXITO — MemoRIA           ║
╠══════════════╦══════════════════════╦═══════════════════════╣
║ Evaluación   ║ Umbral mínimo        ║ Umbral ideal           ║
╠══════════════╬══════════════════════╬═══════════════════════╣
║ E1 Perplejid ║ Mejora ≥ 20% vs base ║ Mejora ≥ 35%          ║
║ E2 Estilo    ║ Diff métricas < 20%  ║ Diff < 10%            ║
║ E3 Clasif.   ║ Accuracy BETO < 75%  ║ Accuracy < 60%        ║
║ E4 Humano    ║ Jueces < 65% correct ║ Jueces < 55% correct  ║
╚══════════════╩══════════════════════╩═══════════════════════╝""")

# Recoger resultados de las celdas anteriores (si corrieron)
results_summary = {
    "E1_perplexity": results.get("improvement_pct") if "results" in dir() else None,
    "E3_classifier_accuracy": acc_result.get("accuracy") if "acc_result" in dir() else None,
}
print("\nResultados del equipo:")
for k, v in results_summary.items():
    print(f"  {k}: {'⏳ pendiente' if v is None else v}")

# Guardar a archivo para el informe
os.makedirs("eval/results", exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
results_file = f"eval/results/{ts}.json"
with open(results_file, "w") as _f:
    json.dump(results_summary, _f, indent=2, ensure_ascii=False)
print(f"\nResultados guardados en {results_file}")

---
## 7. Backend FastAPI <a id='backend'></a>

El backend completo está en `backend/main.py`. El frontend HTML/JS está en `backend/static/`.

```bash
# Arrancar (requiere Ollama corriendo con el modelo memoria)
uvicorn backend.main:app --host 127.0.0.1 --port 8000 --reload
# Abrir: http://127.0.0.1:8000

# O con Docker Compose:
docker compose up
```

In [ ]:
# Verificar que el backend responde (debe estar corriendo primero)
import httpx

try:
    r = httpx.get("http://127.0.0.1:8000/health", timeout=3)
    data = r.json()
    print(f"Backend: {data['status']}")
    print(f"Modelos en Ollama: {data.get('models', [])}")
except Exception as e:
    print(f"Backend no disponible: {e}")
    print("Arrancar con: uvicorn backend.main:app --host 127.0.0.1 --port 8000")

---
## Stack técnico

| Componente | Herramienta |
|-----------|-------------|
| Modelo base | `Qwen/Qwen3-4B-Instruct-2507` |
| Fine-tuning | MLX-LM con cuantización 4-bit + LoRA (rank 16, alpha 32) |
| Inferencia local | Ollama `memoria` (Qwen 3 4B merged + GGUF Q4_K_M) |
| Formato de dataset | `{messages: [system, user+[TAG], assistant]}` (chat mode de MLX-LM) |
| Backend | FastAPI + SSE streaming |
| Frontend | HTML + Vanilla JS + CSS |
| Clasificador de eval | BETO (`dccuchile/bert-base-spanish-wwm-uncased`) |
| Métrica distribucional | MAUVE (`evaluate-metric/mauve`) |